In [1]:
# !git branch


  master
* upload_files


In [3]:
import requests
import importlib
from bs4 import BeautifulSoup
import pandas as pd
import snowflake.connector
import sqlalchemy as db
import numpy as np
import import_ipynb
import re
from datetime import datetime, timedelta
import snowflake_functions as sf  # Import the notebook as a module
import transform_data as td 
import extract_data as ed 
from skills_array import get_skills_array
skills_array = get_skills_array() # get full list of skills

In [5]:
# upload latest function
importlib.reload(sf)
importlib.reload(ed)
importlib.reload(td)

# connection snowflake
engine = sf.connect_snowflake()
connection = engine.connect()

# intialise tables that will later be loaded to snowflake
job_table = []
job_skills_table = []

# go through many pages
max_pages = 1
for j in range(max_pages):
    
    # get all job URL links
    full_urls = ed.get_URLs_to_jobs(j) 
    
    # go through all the URLs and extract relevant data.
    for url in full_urls:
        
        # Scrape job add URL
        response = requests.get(url) 
        soup = BeautifulSoup(response.content, "html.parser") 
        
        # Enter job ad in to jobs table
        job_props, req_skills = ed.get_job_data(soup, skills_array)
        parsed_salary = td.parse_salary(job_props[4]) # clean up salary
        sf.add_to_jobs_table(job_props, parsed_salary, engine)
    
        # get that entries id, get skill ids and enter it into job_skills table.
        job_id = sf.last_job_id(connection)
        df_skill_ids = sf.get_skill_ids(req_skills, connection)
        sf.add_to_job_skills_table(job_id, df_skill_ids, engine)


# Close connection
connection.close()

Connected to Snowflake!
https://www.seek.com.au/job/81948813?type=standard&ref=search-standalone
https://www.seek.com.au/job/82161696?type=standard&ref=search-standalone
https://www.seek.com.au/job/82340497?type=standard&ref=search-standalone
https://www.seek.com.au/job/82356435?type=standard&ref=search-standalone
https://www.seek.com.au/job/82126914?type=standard&ref=search-standalone
https://www.seek.com.au/job/82256211?type=standard&ref=search-standalone
https://www.seek.com.au/job/82299358?type=standard&ref=search-standalone
https://www.seek.com.au/job/82105910?type=standard&ref=search-standalone
https://www.seek.com.au/job/82321927?type=standard&ref=search-standalone
https://www.seek.com.au/job/82294554?type=promoted&ref=search-standalone
https://www.seek.com.au/job/82144319?type=standard&ref=search-standalone
https://www.seek.com.au/job/82297751?type=standard&ref=search-standalone
https://www.seek.com.au/job/82253004?type=standard&ref=search-standalone
https://www.seek.com.au/job

PendingRollbackError: Can't reconnect until invalid transaction is rolled back. (Background on this error at: https://sqlalche.me/e/14/8s2b)

In [ ]:
# Close connection
connection.close()

In [6]:
# Example DataFrame
job_data = {
    'TITLE': job_props[0],
    'COMPANY': job_props[1],
    'LOCATION': job_props[2],
    'EMPLOYMENT_TYPE': job_props[3],
    'SALARY': parsed_salary[0],
    'PAY_PERIOD': parsed_salary[1],
    'POST_DATE': job_props[5]
}
print(job_data)

# job_table_entry = pd.DataFrame([job_data])

{'TITLE': 'Senior Data Engineer', 'COMPANY': None, 'LOCATION': 'Sydney NSW', 'EMPLOYMENT_TYPE': 'Contract/Temp', 'SALARY': 850, 'PAY_PERIOD': 'daily', 'POST_DATE': datetime.date(2025, 2, 11)}
